# NeuroRes-GNN: DGL 2026 Brain Graph Super-Resolution (Spec §3.1)

Spec-compliant deliverable: 3-fold CV with anchor random seed, bar plots of 8 evaluation measures, per-fold prediction CSVs, and Kaggle submission. Primary model: **v3r_eb_ffnn** (best config). Optional comparison with SGC and VGAE baselines.

## 1. Reproducibility and Setup (Spec Note 2)

Insert configurations from basiralab/DGL reproducibility.py at the beginning for deterministic results.

In [1]:
import subprocess
import sys
from pathlib import Path

notebook_dir = Path('.').resolve()
repo_root = notebook_dir.parent
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / 'gcn-encoder-ca-decoder'))

# Reproducibility config (Spec Note 2)
import reproducibility
random_seed = reproducibility.random_seed
DEVICE = reproducibility.device

import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

from data_utils import vec_to_adj, lr_node_features, to_tensor
from models.sgc_baseline import SGCBaseline
from models.vgae import VGAEBaseline
from utils.matrix_vectorizer import MatrixVectorizer
from utils.metrics import evaluate_fold
from utils.plotting import plot_folds

CUDA is available. Using GPU.


/home/jeet/y4/neurores-gnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config

- **Primary model:** v3r_eb_ffnn (best-performing NeuroRes-GNN config)
- **COMPARE_BASELINES:** Set to True to also run SGC and VGAE for comparison (adds runtime)

In [2]:
# Primary model: v3r_eb_ffnn (best config). Set to False to skip SGC/VGAE comparison.
PRIMARY_MODEL = "v3r_eb_ffnn"
COMPARE_BASELINES = True
OUTPUT_DIR = repo_root / "submission"
ARTIFACTS_DIR = repo_root / "artifacts" / "v3r_eb_ffnn"

## 2. Main Entry Function (Spec 3.1.d)

All inputs received as function parameters: train/test data and model parameters.

In [3]:
def run_v3r_eb_ffnn_cv(data_dir: Path, out_dir: Path, artifacts_dir: Path) -> dict:
    """Run best config (v3r_eb_ffnn) via train_dense_gcn script. Load artifacts, plot, copy to submission."""
    import json
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-m", "src.train_dense_gcn", "cv", "--preset", "v3r_eb_ffnn",
           "--data-dir", str(data_dir), "--out-dir", str(artifacts_dir), "--fresh"]
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, check=True, cwd=str(repo_root))
    with open(artifacts_dir / "cv_summary.json") as f:
        cv = json.load(f)
    fold_metrics = [r["metrics"] for r in cv["folds"]]
    plot_folds(fold_metrics)
    for i in (1, 2, 3):
        src = artifacts_dir / f"predictions_fold_{i}.csv"
        if src.exists():
            pd.read_csv(src).to_csv(out_dir / f"predictions_fold_{i}.csv", index=False)
    sub_src = artifacts_dir / "submission.csv"
    if sub_src.exists():
        pd.read_csv(sub_src).to_csv(out_dir / "submission.csv", index=False)
    print(f"Copied outputs to {out_dir}")
    return {"fold_metrics": fold_metrics}


def run_3fold_cv_sgc(
    X_lr_train: np.ndarray,
    Y_hr_train: np.ndarray,
    X_lr_test: np.ndarray,
    n_lr: int = 160,
    n_hr: int = 268,
    model_kwargs: dict | None = None,
    epochs: int = 400,
    patience: int = 30,
    batch_size: int = 16,
    lr: float = 1e-3,
    weight_decay: float = 1e-5,
    random_state: int = 42,
    output_dir: Path | str | None = None,
    plot: bool = True,
) -> dict:
    """
    Run 3-fold CV with SGC, evaluate on 8 metrics, save predictions_fold_{1,2,3}.csv,
    generate bar plots (if plot=True), and produce final submission.

    Parameters
    ----------
    X_lr_train : np.ndarray
        LR training data (167, 12720).
    Y_hr_train : np.ndarray
        HR training labels (167, 35778).
    X_lr_test : np.ndarray
        LR test data (112, 12720).
    n_lr, n_hr : int
        LR/HR node counts.
    model_kwargs : dict
        Arguments for SGCBaseline (n_lr, n_hr, d_model, K, in_node_feat_dim).
    epochs, patience, batch_size, lr, weight_decay : float/int
        Training hyperparameters.
    random_state : int
        Anchor seed for KFold (Spec Note 3).
    output_dir : Path | str
        Directory for predictions_fold_*.csv and submission.csv.

    Returns
    -------
    dict
        fold_metrics, fold_models, test_preds, ensemble.
    """
    if model_kwargs is None:
        model_kwargs = dict(n_lr=n_lr, n_hr=n_hr, d_model=64, K=2, in_node_feat_dim=2)
    else:
        model_kwargs = {**dict(n_lr=n_lr, n_hr=n_hr), **model_kwargs}

    out = Path(output_dir) if output_dir else repo_root / 'submission'
    out.mkdir(parents=True, exist_ok=True)

    # KFold with shuffle=True and random_state (Spec Note 3)
    kf = KFold(n_splits=3, shuffle=True, random_state=random_state)
    vectorizer = MatrixVectorizer()

    fold_metrics = []
    fold_models = []
    test_preds = []

    for fold_id, (tr_idx, va_idx) in enumerate(kf.split(X_lr_train), start=1):
        print(f"\n{'='*50}")
        print(f'Fold {fold_id}: train={len(tr_idx)}, val={len(va_idx)}')
        print(f"{'='*50}")

        model = _train_one_fold(
            X_lr_train[tr_idx], Y_hr_train[tr_idx],
            X_lr_train[va_idx], Y_hr_train[va_idx],
            fold_id, model_kwargs, epochs, patience, batch_size, lr, weight_decay,
        )
        fold_models.append(model)

        # Evaluate on validation set (for bar plots)
        pred_vecs, gt_vecs = _predict_validation(model, X_lr_train[va_idx], Y_hr_train[va_idx], n_lr, n_hr, batch_size)
        pred_mats = np.stack([vectorizer.anti_vectorize(v, n_hr) for v in pred_vecs])
        gt_mats = np.stack([vectorizer.anti_vectorize(v, n_hr) for v in gt_vecs])
        metrics = evaluate_fold(pred_mats, gt_mats, verbose=True)
        fold_metrics.append(metrics)

        # Test predictions from this fold's model (Spec: predictions_fold_{fold_num}.csv)
        test_pred = _predict_test(model, X_lr_test, n_lr, batch_size)
        test_preds.append(test_pred)

        # Save predictions_fold_{fold_num}.csv in Kaggle format (Spec 3.1.1)
        _save_predictions_csv(test_pred, out / f'predictions_fold_{fold_id}.csv')
        print(f'  Saved {out / f"predictions_fold_{fold_id}.csv"}')

    # Bar plots of 8 metrics (Spec 3.1.2)
    if plot:
        plot_folds(fold_metrics)

    # Final submission: ensemble of 3 folds, post-processed (no negatives, Spec Note 4)
    ensemble = np.clip(np.mean(test_preds, axis=0), a_min=0.0, a_max=None)
    _save_predictions_csv(ensemble, out / 'submission.csv')
    print(f'\nSaved submission: {out / "submission.csv"}')

    return {'fold_metrics': fold_metrics, 'fold_models': fold_models, 'test_preds': test_preds, 'ensemble': ensemble}


def _train_one_fold(X_tr, Y_tr, X_va, Y_va, fold_id, model_kwargs, epochs, patience, batch_size, lr, weight_decay):
    """Train model for one fold with early stopping."""
    Xtr, Ytr = to_tensor(X_tr, DEVICE), to_tensor(Y_tr, DEVICE)
    Xva, Yva = to_tensor(X_va, DEVICE), to_tensor(Y_va, DEVICE)
    tr_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=batch_size, shuffle=False)

    n_lr = model_kwargs['n_lr']
    model = SGCBaseline(**model_kwargs).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val, best_state = float('inf'), None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for x_vec, y_vec in tr_loader:
            A = vec_to_adj(x_vec, n_lr)
            X = lr_node_features(A)
            loss = loss_fn(model(A, X), y_vec)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_vec, y_vec in va_loader:
                A = vec_to_adj(x_vec, n_lr)
                X = lr_node_features(A)
                val_losses.append(loss_fn(model(A, X), y_vec).item())
        val_loss = np.mean(val_losses)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 50 == 0 or epoch == 1:
            print(f'  Fold {fold_id} | Epoch {epoch:3d}/{epochs} | Val: {val_loss:.6f}')

        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model


def _predict_validation(model, X_va, Y_va, n_lr, n_hr, batch_size):
    """Predict on validation set for metric evaluation."""
    model.eval()
    loader = DataLoader(TensorDataset(to_tensor(X_va, DEVICE), to_tensor(Y_va, DEVICE)), batch_size=batch_size, shuffle=False)
    preds, gts = [], []
    with torch.no_grad():
        for x_vec, y_vec in loader:
            A = vec_to_adj(x_vec, n_lr)
            X = lr_node_features(A)
            preds.append(model(A, X).cpu().numpy())
            gts.append(y_vec.cpu().numpy())
    return np.concatenate(preds), np.concatenate(gts)


def _predict_test(model, X_test, n_lr, batch_size):
    """Predict on test set (112 subjects)."""
    model.eval()
    loader = DataLoader(TensorDataset(to_tensor(X_test, DEVICE)), batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (x_vec,) in loader:
            A = vec_to_adj(x_vec, n_lr)
            X = lr_node_features(A)
            preds.append(model(A, X).cpu().numpy())
    return np.concatenate(preds)


def _save_predictions_csv(preds: np.ndarray, path: Path):
    """Save predictions in Kaggle format: ID, Predicted (Spec 3.1.1)."""
    preds = np.clip(preds, a_min=0.0, a_max=None)  # No negatives (Spec Note 4)
    n_subjects, n_features = preds.shape
    ids = np.arange(1, n_subjects * n_features + 1)
    pd.DataFrame({'ID': ids, 'Predicted': preds.flatten()}).to_csv(path, index=False)


def run_3fold_cv_vgae(X_lr_train, Y_hr_train, X_lr_test, n_lr=160, n_hr=268, epochs=50, patience=15,
                      batch_size=16, lr=1e-3, weight_decay=1e-5, random_state=42, output_dir=None) -> dict:
    """Run 3-fold CV with VGAE baseline. Returns fold_metrics for comparison."""
    out = Path(output_dir) if output_dir else repo_root / "submission"
    out.mkdir(parents=True, exist_ok=True)
    kf = KFold(n_splits=3, shuffle=True, random_state=random_state)
    vectorizer = MatrixVectorizer()
    fold_metrics, test_preds = [], []
    vgae_kw = dict(n_lr=n_lr, n_hr=n_hr, hidden_dim=64, latent_dim=64, in_node_feat_dim=2, dropout=0.1)
    for fold_id, (tr_idx, va_idx) in enumerate(kf.split(X_lr_train), start=1):
        model = VGAEBaseline(**vgae_kw).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        Xtr, Ytr = to_tensor(X_lr_train[tr_idx], DEVICE), to_tensor(Y_hr_train[tr_idx], DEVICE)
        Xva, Yva = to_tensor(X_lr_train[va_idx], DEVICE), to_tensor(Y_hr_train[va_idx], DEVICE)
        tr_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
        va_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=batch_size, shuffle=False)
        best_val, best_state = float('inf'), None
        pc = 0
        for ep in range(1, epochs + 1):
            model.train()
            for x, y in tr_loader:
                A = vec_to_adj(x, n_lr)
                X = lr_node_features(A)
                loss = nn.MSELoss()(model(A, X), y)
                opt.zero_grad()
                loss.backward()
                opt.step()
            model.eval()
            val_losses = []
            with torch.no_grad():
                for x, y in va_loader:
                    A = vec_to_adj(x, n_lr)
                    X = lr_node_features(A)
                    val_losses.append(nn.MSELoss()(model(A, X), y).item())
            val_loss = np.mean(val_losses)
            if val_loss < best_val:
                best_val, best_state, pc = val_loss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
            else:
                pc += 1
            if pc >= patience:
                break
        model.load_state_dict(best_state)
        pred_vecs, gt_vecs = _predict_validation(model, X_lr_train[va_idx], Y_hr_train[va_idx], n_lr, n_hr, batch_size)
        pred_mats = np.stack([vectorizer.anti_vectorize(v, n_hr) for v in pred_vecs])
        gt_mats = np.stack([vectorizer.anti_vectorize(v, n_hr) for v in gt_vecs])
        metrics = evaluate_fold(pred_mats, gt_mats, verbose=True)
        fold_metrics.append(metrics)
        test_pred = _predict_test(model, X_lr_test, n_lr, batch_size)
        test_preds.append(test_pred)
        _save_predictions_csv(test_pred, out / f"predictions_fold_{fold_id}_vgae.csv")
    return {"fold_metrics": fold_metrics, "test_preds": test_preds}

## 3. Load Data and Run (Single Run - Spec 3.1.c)

Load ALR_tr, AHR_tr (167 train), and test LR. Run 3-fold CV; outputs: predictions_fold_{1,2,3}.csv, bar plots, submission.csv.

In [ ]:
def load_csv(path):
    """Load competition CSV (skip header row)."""
    arr = np.loadtxt(path, delimiter=',', skiprows=1, dtype=np.float32)
    return arr if arr.ndim > 1 else arr[None, :]

# Load data
X_lr_train = load_csv(repo_root / 'data' / 'lr_train.csv')
Y_hr_train = load_csv(repo_root / 'data' / 'hr_train.csv')
X_lr_test = load_csv(repo_root / 'data' / 'lr_test.csv')
data_dir = repo_root / 'data'
print(f'Train LR: {X_lr_train.shape} | Train HR: {Y_hr_train.shape} | Test LR: {X_lr_test.shape}')

# 1. Run primary model (v3r_eb_ffnn) - best config
result_v3r = run_v3r_eb_ffnn_cv(data_dir, OUTPUT_DIR, ARTIFACTS_DIR)

# 2. Optional: Compare with SGC and VGAE baselines (saves to artifacts/comparison_* to avoid overwriting)
if COMPARE_BASELINES:
    comp_dir = repo_root / "artifacts" / "comparison"
    print('\n' + '='*60 + '\nRunning SGC baseline for comparison...\n' + '='*60)
    sgc_kw = dict(n_lr=160, n_hr=268, d_model=64, K=2, in_node_feat_dim=2)
    result_sgc = run_3fold_cv_sgc(X_lr_train, Y_hr_train, X_lr_test, model_kwargs=sgc_kw,
        epochs=400, patience=30, random_state=random_seed, output_dir=comp_dir / "sgc", plot=False)
    print('\n' + '='*60 + '\nRunning VGAE baseline for comparison...\n' + '='*60)
    result_vgae = run_3fold_cv_vgae(X_lr_train, Y_hr_train, X_lr_test, epochs=50, patience=15,
        random_state=random_seed, output_dir=comp_dir / "vgae")
    # Comparison table (mean MAE across folds)
    from utils.plotting import summarize_folds
    m_sgc, _ = summarize_folds(result_sgc['fold_metrics'])
    m_vgae, _ = summarize_folds(result_vgae['fold_metrics'])
    m_v3r, _ = summarize_folds(result_v3r['fold_metrics'])
    print('\n--- Comparison (mean MAE) ---')
    print(f"NeuroRes-GNN (v3r_eb_ffnn): {m_v3r['MAE']:.6f}")
    print(f"SGC baseline:              {m_sgc['MAE']:.6f}")
    print(f"VGAE baseline:             {m_vgae['MAE']:.6f}")

print('\nDone. Outputs: predictions_fold_{1,2,3}.csv, submission.csv')

Train LR: (167, 12720) | Train HR: (167, 35778) | Test LR: (112, 12720)
Running: /home/jeet/y4/neurores-gnn/.venv/bin/python -m src.train_dense_gcn cv --preset v3r_eb_ffnn --data-dir /home/jeet/y4/neurores-gnn/data --out-dir /home/jeet/y4/neurores-gnn/artifacts/v3r_eb_ffnn --fresh
Device: cuda
Train LR: (167, 12720) | Train HR: (167, 35778)
Config: {'model_name': 'dense_gcn', 'n_lr': 160, 'n_hr': 268, 'hidden_dim': 192, 'num_layers': 3, 'dropout': 0.35, 'epochs': 400, 'patience': 60, 'batch_size': 16, 'learning_rate': 0.0008, 'weight_decay': 0.0001, 'loss_name': 'l1', 'huber_beta': 1.0, 'l1_weight': 0.7, 'seed': 42, 'num_folds': 3, 'num_heads': 4, 'ffn_mult': 4, 'num_decoder_heads': 4, 'hr_refine_layers': 1, 'edge_scale': 0.2, 'bipartite_layers': 1, 'max_epochs_full': 50, 'use_residual': True, 'mixup_alpha': 0.2, 'subject_scale': False, 'calibrate': False, 'lap_pe_dim': 0, 'pearl_pe_dim': 0, 'lr_schedule': 'plateau', 'lr_min': 1e-06, 'curriculum_phase_epochs': 0, 'curriculum_heavy_perc

Graph metrics:  21%|██▏       | 12/56 [04:51<20:08, 27.46s/it]